# Dataset preview — Drosophila + zebrafish (unlabeled, from Sophie)
Extract short 20-frame preview videos (mp4) from the supervisor's calcium-imaging data.
Attach the Kaggle dataset `mokarbalaee/neuroseg-drosophila-larvae`, run all cells,
then download the mp4s from `/kaggle/working/previews/`.

In [ ]:
!pip install -q czifile imageio imageio-ffmpeg

## 1. List the video files (check the mount path)

In [ ]:
from pathlib import Path
print('input dirs:', [str(p) for p in Path('/kaggle/input').glob('*')])

# EDIT this if the printed mount path differs:
DATA = Path('/kaggle/input/neuroseg-drosophila-larvae')

exts = ('.tif', '.tiff', '.czi', '.sec')
files = sorted(p for p in DATA.rglob('*') if p.suffix.lower() in exts)
for f in files:
    print(f'{f.stat().st_size/1e6:8.1f} MB   {f.relative_to(DATA)}')
print(f'\n{len(files)} video files found')

## 2. Helpers: read frames (TIFF / CZI / .sec), split ETL planes, write mp4

In [ ]:
import numpy as np, tifffile as tiff, imageio.v2 as imageio

def read_n_frames(path, n=20):
    """Read ~n evenly spaced frames from a TIFF/CZI/.sec calcium video (memory-safe)."""
    path = str(path); ext = Path(path).suffix.lower()
    if ext in ('.czi', '.sec'):
        import czifile
        arr = np.squeeze(czifile.imread(path))
        if arr.ndim > 3:
            arr = arr.reshape((-1,) + arr.shape[-2:])
        if arr.ndim == 2:
            arr = arr[None]
        idx = np.linspace(0, arr.shape[0]-1, min(n, arr.shape[0])).astype(int)
        return arr[idx].astype(np.float32)
    with tiff.TiffFile(path) as tf:
        pages = tf.pages
        if len(pages) > 1:                       # multi-page stack: read only n pages
            idx = np.linspace(0, len(pages)-1, min(n, len(pages))).astype(int)
            return np.stack([pages[int(i)].asarray() for i in idx]).astype(np.float32)
        arr = tf.asarray()                        # single-page: whole array
    if arr.ndim == 2:
        arr = arr[None]
    idx = np.linspace(0, arr.shape[0]-1, min(n, arr.shape[0])).astype(int)
    return arr[idx].astype(np.float32)

def split_planes(frames):
    """Split ETL side-by-side multi-plane frames (W = k*H) into k separate planes."""
    T, H, W = frames.shape
    if W > H and W % H == 0 and W // H > 1:
        k = W // H
        return [frames[:, :, i*H:(i+1)*H] for i in range(k)]
    return [frames]

def write_video(frames, out, fps=8):
    """Global-normalize (fixed contrast across frames) and write an mp4."""
    lo, hi = np.percentile(frames, 1), np.percentile(frames, 99.7)
    vid = np.clip((frames - lo) / (hi - lo + 1e-8), 0, 1)
    vid = (vid * 255).astype(np.uint8)
    vid = np.stack([np.stack([f]*3, -1) for f in vid])   # gray -> RGB
    imageio.mimsave(str(out), list(vid), fps=fps, codec='libx264')
    print('wrote', out, 'shape', vid.shape)

## 3. Render 20-frame previews
Auto-picks one Drosophila CZI, one Drosophila TIFF, and the smallest zebrafish ETL file.
Edit `picks` after seeing the file list above if you want specific files.

In [ ]:
OUT = Path('/kaggle/working/previews'); OUT.mkdir(parents=True, exist_ok=True)

czis     = [f for f in files if f.suffix.lower() in ('.czi', '.sec')]
dros_tif = [f for f in files if f.suffix.lower() in ('.tif', '.tiff') and 'etl' not in f.name.lower()]
etl      = sorted([f for f in files if 'etl' in f.name.lower()], key=lambda f: f.stat().st_size)

picks = []
if czis:     picks.append(('drosophila_czi', czis[0]))
if dros_tif: picks.append(('drosophila_tif', dros_tif[0]))
if etl:      picks.append(('zebrafish_etl',  etl[0]))   # smallest ETL (avoids the 15.8 GB file)

for tag, f in picks:
    print('reading', f.name)
    frames = read_n_frames(f, n=20)
    planes = split_planes(frames)
    plane = planes[len(planes)//2]        # middle plane for ETL zebrafish; whole frame otherwise
    print(f'  {tag}: {frames.shape} -> {len(planes)} plane(s), using {plane.shape}')
    write_video(plane, OUT / f'{tag}.mp4', fps=8)

import os
print('\nDone. Download these from', OUT, ':', os.listdir(OUT))